<a href="https://colab.research.google.com/github/ffahrialfikri/Data-Science-2026/blob/main/pertemuan12_fikrialfahri_250401020144.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama: Fikri Alfahri
Kelas: IF405
NIM: 250401020144

In [1]:
#Langkah 1: Generate & Eksplorasi Dataset Transaksi
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:', transaksi[:3])
print('Jumlah transaksi:', len(transaksi))


#Langkah 2: One-Hot Encoding Transaksi
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print(df.head())


#Langkah 3: Cari Frequent Itemset dengan Apriori
from mlxtend.frequent_patterns import apriori

# Uji beberapa nilai min_support
for ms in [0.05, 0.1, 0.2]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print("\nTop 10 Frequent Itemsets:")
print(freq_items.head(10))

#Langkah 4: Bentuk & Saring Aturan Asosiasi
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence', min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))


#Langkah 5: Rekomender Sederhana dengan Content-Based Filtering
from sklearn.metrics.pairwise import cosine_similarity

# 1. Buat katalog produk dengan kategori
katalog = pd.DataFrame({
    'produk': produk,
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy',
                 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

# 2. One-hot encoding untuk kategori dan hitung cosine similarity
fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

# 3. Fungsi rekomendasi
def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]
    return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))


#Langkah 6: Bandingkan Kedua Pendekatan
produk_target = 'Roti'

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung produk_target
rules_terkait = rules[rules['antecedents'].apply(lambda x: produk_target in x)]

print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())

print('\nRekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

# Diskusi Singkat:
"""
- Association Rules merekomendasikan item berdasarkan kebiasaan transaksi nyata (misal: Roti -> Selai).
- Content-Based Filtering merekomendasikan item dari kategori/fitur serupa (misal: Roti -> produk Bakery lain).
- Kedua pendekatan ini idealnya digabungkan (Hybrid) untuk mengatasi kekurangan satu sama lain, seperti masalah cold start pada transaksi.
"""

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50
    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False
min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan

Top 10 Frequent Itemsets:
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34   

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']
Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115

Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


'\n- Association Rules merekomendasikan item berdasarkan kebiasaan transaksi nyata (misal: Roti -> Selai).\n- Content-Based Filtering merekomendasikan item dari kategori/fitur serupa (misal: Roti -> produk Bakery lain).\n- Kedua pendekatan ini idealnya digabungkan (Hybrid) untuk mengatasi kekurangan satu sama lain, seperti masalah cold start pada transaksi.\n'

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


### Kesimpulan Alur Kerja

1.  **Langkah 1 (Data Generation):** Menyiapkan dataset sintetis berisi 50 transaksi dengan pola belanja spesifik (seperti hubungan Roti dan Selai).
2.  **Langkah 2 (Preprocessing):** Mengubah data transaksi menjadi format matriks biner (One-Hot Encoding) agar dapat diproses oleh algoritma machine learning.
3.  **Langkah 3 (Frequent Itemsets):** Menggunakan algoritma **Apriori** untuk menemukan produk atau kombinasi produk yang paling sering muncul berdasarkan ambang batas *support*.
4.  **Langkah 4 (Association Rules):** Membentuk aturan "jika-maka" (misal: Jika beli Roti, maka beli Selai) dan menyaringnya berdasarkan nilai *confidence* dan *lift*.
5.  **Langkah 5 (Content-Based Filtering):** Membangun sistem rekomendasi berdasarkan kesamaan kategori produk menggunakan *Cosine Similarity*.
6.  **Langkah 6 (Perbandingan):** Membandingkan hasil rekomendasi. *Association Rules* bersifat behavioral (berdasarkan data nyata), sedangkan *Content-Based* bersifat atributif (berdasarkan kesamaan jenis produk).

### Library yang Digunakan

Dalam notebook ini, pustaka-pustaka Python berikut digunakan:

*   **`pandas`**: Manipulasi dan analisis data dalam bentuk DataFrame.
*   **`numpy`**: Komputasi numerik dan pembuatan data acak.
*   **`matplotlib`**: Visualisasi data (opsional untuk pengembangan lanjut).
*   **`mlxtend`**: Digunakan untuk implementasi algoritma **Apriori** dan pembentukan **Association Rules**.
*   **`scikit-learn` (sklearn)**: Digunakan untuk menghitung **Cosine Similarity** pada sistem rekomendasi *Content-Based*.